In [ ]:

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from sentence_transformers import SentenceTransformer
from transformers import AutoModel, AutoTokenizer
from peft import LoraConfig, get_peft_model
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from imblearn.over_sampling import SMOTE

import random
torch.manual_seed(7)
np.random.seed(7)
random.seed(7)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False



device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


file_path = 'Java1.csv'
data = pd.read_csv(file_path)


data['concatenated_text'] = data['Processed Body'] + " " + data['Tags'] + " " + data['Title']


label_mapping = {'Basic': 0, 'Intermediate': 1, 'Advanced': 2}
data['encoded_labels'] = data['Label'].map(label_mapping)
labels = data['encoded_labels'].values

numerical_cols = [
    'User_Reputation', 'Bronze_badge', 'Gold_badge', 'Silver_badge',
    'User Id', 'Accepted Answer ID', 'View Count', 'Answer Count',
    'Score', 'Interval from first', 'Interval from accepted',
    'Total count urls and imgs', 'LOC', 'Question_Length'
]


combined_texts = data['concatenated_text'].tolist()
numerical_data = data[numerical_cols].values
labels = data['Label'].map(label_mapping).values  

X_text_train, X_text_test, X_num_train, X_num_test, y_train, y_test = train_test_split(
    combined_texts, numerical_data, labels,
    test_size=0.20, random_state=42, stratify=labels
)

imputer = SimpleImputer(strategy='mean')
X_num_train = imputer.fit_transform(X_num_train)
X_num_test = imputer.transform(X_num_test)

scaler = StandardScaler()
X_num_train = scaler.fit_transform(X_num_train)
X_num_test = scaler.transform(X_num_test)


model_name = "sentence-transformers/all-MiniLM-L6-v2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
base_model = AutoModel.from_pretrained(model_name)

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=["query", "value"]
)
model = get_peft_model(base_model, lora_config)
model.to(device)

class StackOverflowDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts = texts
        self.labels = labels

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        inputs = tokenizer(
            self.texts[idx],
            padding="max_length",
            truncation=True,
            return_tensors="pt",
            max_length=512
        )
        if "token_type_ids" in inputs:
            del inputs["token_type_ids"]
        return {
            "input_ids": inputs["input_ids"].squeeze(0),
            "attention_mask": inputs["attention_mask"].squeeze(0),
            "label": torch.tensor(self.labels[idx], dtype=torch.long)
        }

train_dataset = StackOverflowDataset(X_text_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)

class SBERTWithClassification(nn.Module):
    def __init__(self, base_model, num_classes):
        super(SBERTWithClassification, self).__init__()
        self.base_model = base_model
        self.classifier = nn.Linear(384, num_classes)

    def forward(self, input_ids, attention_mask):
        outputs = self.base_model(input_ids=input_ids, attention_mask=attention_mask)
        cls_embedding = outputs.last_hidden_state[:, 0, :]
        logits = self.classifier(cls_embedding)
        return logits

num_classes = len(label_mapping)
classifier_model = SBERTWithClassification(model, num_classes).to(device)

loss_fn = nn.CrossEntropyLoss()
optimizer = optim.AdamW(classifier_model.parameters(), lr=5e-5)

num_epochs = 3
for epoch in range(num_epochs):
    classifier_model.train()
    total_loss = 0
    for batch in train_loader:
        optimizer.zero_grad()
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)
        outputs = classifier_model(input_ids=input_ids, attention_mask=attention_mask)
        loss = loss_fn(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {total_loss / len(train_loader):.4f}")

def extract_embeddings(texts):
    inputs = tokenizer(texts, padding=True, truncation=True, return_tensors="pt", max_length=512).to(device)
    if "token_type_ids" in inputs:
        del inputs["token_type_ids"]
    with torch.no_grad():
        outputs = classifier_model.base_model(**inputs)
    return outputs.last_hidden_state[:, 0, :].cpu().numpy()

train_embeddings = np.array([extract_embeddings([text])[0] for text in X_text_train])
test_embeddings = np.array([extract_embeddings([text])[0] for text in X_text_test])

X_train_combined = np.hstack((train_embeddings, X_num_train))
X_test_combined = np.hstack((test_embeddings, X_num_test))

smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train_combined, y_train)

clf = RandomForestClassifier(
    n_estimators=400, max_depth=20, bootstrap=True,
    max_features='sqrt', min_samples_split=10,
    min_samples_leaf=1, random_state=42
)
clf.fit(X_train_resampled, y_train_resampled)

y_pred = clf.predict(X_test_combined)
print(classification_report(y_test, y_pred, target_names=label_mapping.keys(), digits=4))


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report

param_grid = {
    'n_estimators': [100, 200, 400],
    'max_depth': [10, 20, 30],
    'bootstrap': [True, False],
    'max_features': ['sqrt', 'log2'],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
}

rf = RandomForestClassifier(random_state=42)

grid_search = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    cv=5,
    n_jobs=-1,
    verbose=2,
    scoring='f1_macro'  
)

grid_search.fit(X_train_resampled, y_train_resampled)

best_rf = grid_search.best_estimator_
print("Best Parameters:", grid_search.best_params_)

y_pred = best_rf.predict(X_test_combined)
print(classification_report(y_test, y_pred, target_names=label_mapping.keys(), digits=4))


Fitting 5 folds for each of 324 candidates, totalling 1620 fits


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The 

[CV] END bootstrap=True, max_depth=10, max_features=sqrt, min_samples_leaf=1, min_samples_split=5, n_estimators=100; total time=   1.5s
[CV] END bootstrap=True, max_depth=10, max_features=sqrt, min_samples_leaf=1, min_samples_split=2, n_estimators=100; total time=   1.5s
[CV] END bootstrap=True, max_depth=10, max_features=sqrt, min_samples_leaf=1, min_samples_split=5, n_estimators=100; total time=   1.5s
[CV] END bootstrap=True, max_depth=10, max_features=sqrt, min_samples_leaf=1, min_samples_split=2, n_estimators=100; total time=   1.5s
[CV] END bootstrap=True, max_depth=10, max_features=sqrt, min_samples_leaf=1, min_samples_split=5, n_estimators=100; total time=   1.5s
[CV] END bootstrap=True, max_depth=10, max_features=sqrt, min_samples_leaf=1, min_samples_split=5, n_estimators=100; total time=   1.5s
[CV] END bootstrap=True, max_depth=10, max_features=sqrt, min_samples_leaf=1, min_samples_split=5, n_estimators=100; total time=   1.5s
[CV] END bootstrap=True, max_depth=10, max_featu

In [ ]:
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score
import lightgbm as lgb
import xgboost as xgb

results = {}

models_and_params = {
    'RandomForest': {
        'model': RandomForestClassifier(random_state=42),
        'params': {
            'n_estimators': [100, 200, 300],
            'max_depth': [10, 20, 30, None],
            'min_samples_split': [2, 5, 10],
            'min_samples_leaf': [1, 2, 4],
            'max_features': ['sqrt', 'log2'],
            'bootstrap': [True, False]
        }
    },
    'ExtraTrees': {
        'model': ExtraTreesClassifier(random_state=42),
        'params': {
            'n_estimators': [100, 200, 300],
            'max_depth': [10, 20, 30, None],
            'min_samples_split': [2, 5, 10],
            'min_samples_leaf': [1, 2, 4],
            'max_features': ['sqrt', 'log2']
        }
    },
    'LightGBM': {
        'model': lgb.LGBMClassifier(random_state=42),
        'params': {
            'n_estimators': [100, 200, 300],
            'learning_rate': [0.01, 0.05, 0.1],
            'max_depth': [-1, 10, 20, 30],
            'num_leaves': [31, 50, 100],
            'min_child_samples': [20, 40, 60],
            'subsample': [0.8, 1.0],
            'colsample_bytree': [0.8, 1.0]
        }
    },
    'XGBoost': {
        'model': xgb.XGBClassifier(use_label_encoder=False, eval_metric='mlogloss', random_state=42),
        'params': {
            'n_estimators': [100, 200, 300],
            'learning_rate': [0.01, 0.05, 0.1],
            'max_depth': [3, 6, 10],
            'subsample': [0.7, 0.8, 1.0],
            'colsample_bytree': [0.7, 0.8, 1.0],
            'gamma': [0, 1, 5],
            'reg_alpha': [0, 0.5, 1],
            'reg_lambda': [1, 1.5, 2]
        }
    }
}

for name, mp in models_and_params.items():
    print(f"\n🔍 Running GridSearchCV for {name}...")
    grid = GridSearchCV(
        estimator=mp['model'],
        param_grid=mp['params'],
        scoring='accuracy',
        cv=5,
        verbose=2,
        n_jobs=-1
    )
    grid.fit(X_train_resampled, y_train_resampled)
    best_model = grid.best_estimator_
    y_pred = best_model.predict(X_test_combined)
    acc = accuracy_score(y_test, y_pred)

    results[name] = {
        'best_params': grid.best_params_,
        'accuracy': acc
    }

print("\n✅ Grid Search Results Summary:")
for name, res in results.items():
    print(f"\n{name}:")
    print(f"  Best Parameters: {res['best_params']}")
    print(f"  Accuracy: {res['accuracy']:.4f}")


In [ ]:
from sklearn.ensemble import ExtraTreesClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.model_selection import GridSearchCV

classifiers = {
    "RandomForest": (RandomForestClassifier(random_state=42), {
        'n_estimators': [200, 400],
        'max_depth': [10, 20],
        'max_features': ['sqrt'],
        'min_samples_split': [5, 10],
        'min_samples_leaf': [1, 2]
    }),
    "ExtraTrees": (ExtraTreesClassifier(random_state=42), {
        'n_estimators': [200, 400],
        'max_depth': [10, 20],
        'max_features': ['sqrt'],
        'min_samples_split': [5, 10],
        'min_samples_leaf': [1, 2]
    }),
    "XGBoost": (XGBClassifier(use_label_encoder=False, eval_metric='mlogloss', random_state=42), {
        'n_estimators': [200, 400],
        'max_depth': [3, 6],
        'learning_rate': [0.05, 0.1]
    }),
    "LightGBM": (LGBMClassifier(random_state=42), {
        'n_estimators': [200, 400],
        'num_leaves': [31, 50],
        'learning_rate': [0.05, 0.1]
    })
}

from sklearn.metrics import classification_report

for name, (clf, param_grid) in classifiers.items():
    print(f"\n🔍 Running GridSearchCV for {name}...")
    grid = GridSearchCV(clf, param_grid, cv=3, n_jobs=-1, scoring='f1_weighted')
    grid.fit(X_train_resampled, y_train_resampled)

    best_model = grid.best_estimator_
    y_pred = best_model.predict(X_test_combined)

    print(f"\n✅ Classification Report for {name} (Best Params: {grid.best_params_}):")
    print(classification_report(y_test, y_pred, target_names=label_mapping.keys(), digits=4))



🔍 Running GridSearchCV for RandomForest...


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The 


✅ Classification Report for RandomForest (Best Params: {'max_depth': 20, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 5, 'n_estimators': 400}):
              precision    recall  f1-score   support

       Basic     0.7534    0.8871    0.8148       124
Intermediate     0.6415    0.5000    0.5620        68
    Advanced     0.5000    0.2941    0.3704        17

    accuracy                         0.7129       209
   macro avg     0.6316    0.5604    0.5824       209
weighted avg     0.6964    0.7129    0.6964       209


🔍 Running GridSearchCV for ExtraTrees...

✅ Classification Report for ExtraTrees (Best Params: {'max_depth': 20, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 5, 'n_estimators': 400}):
              precision    recall  f1-score   support

       Basic     0.7115    0.8952    0.7929       124
Intermediate     0.6170    0.4265    0.5043        68
    Advanced     0.6667    0.2353    0.3478        17

    accuracy             

/home/aman/jupyter-env/lib/python3.12/site-packages/xgboost/training.py:183: UserWarning: [13:53:10] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/home/aman/jupyter-env/lib/python3.12/site-packages/xgboost/training.py:183: UserWarning: [13:53:10] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/home/aman/jupyter-env/lib/python3.12/site-packages/xgboost/training.py:183: UserWarning: [13:53:10] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/home/aman/jupyter-env/lib/python3.12/site-packages/xgboost/training.py:183: UserWarning: [13:53:10] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/home/aman/jupyter-env/lib/python3.12/site-packages/


✅ Classification Report for XGBoost (Best Params: {'learning_rate': 0.1, 'max_depth': 6, 'n_estimators': 200}):
              precision    recall  f1-score   support

       Basic     0.7698    0.8629    0.8137       124
Intermediate     0.5370    0.4265    0.4754        68
    Advanced     0.2500    0.2353    0.2424        17

    accuracy                         0.6699       209
   macro avg     0.5189    0.5082    0.5105       209
weighted avg     0.6518    0.6699    0.6572       209


🔍 Running GridSearchCV for LightGBM...
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.022235 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 101330
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.025971 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 101396
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead

...

/home/aman/jupyter-env/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

/home/aman/jupyter-env/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/aman/jupyter-env/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

/home/aman/jupyter-env/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
